# Paper Trading Analysis
Analysis of the Kalshi vs Deribit arbitrage strategy performance

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from collections import defaultdict

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## Load Data

In [ ]:
# Load state data
with open('../data/paper_trading/state.json') as f:
    state = json.load(f)

# Load equity curve
with open('../data/paper_trading/equity_curve.json') as f:
    equity_data = json.load(f)

# Load trade log
with open('../data/paper_trading/trade_log.json') as f:
    trade_log = json.load(f)

# Convert positions to DataFrame
positions = pd.DataFrame(state['positions'].values())
positions['entry_time'] = pd.to_datetime(positions['entry_time'])
positions['exit_time'] = pd.to_datetime(positions['exit_time'])
positions['settlement_time'] = pd.to_datetime(positions['settlement_time'])

# Separate open and closed positions
closed = positions[positions['exit_price'].notna()].copy()
open_pos = positions[positions['exit_price'].isna()].copy()

print(f"Total positions: {len(positions)}")
print(f"Closed: {len(closed)}")
print(f"Open: {len(open_pos)}")
print(f"\nCurrent bankroll: ${state['bankroll']:,.2f}")
print(f"Peak bankroll: ${state['peak_bankroll']:,.2f}")

## Equity Curve

In [ ]:
# Convert equity curve to DataFrame
equity_df = pd.DataFrame(equity_data)
equity_df['timestamp'] = pd.to_datetime(equity_df['timestamp'])
equity_df['return_pct'] = (equity_df['bankroll'] / 10000 - 1) * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Equity curve
ax1 = axes[0]
ax1.plot(equity_df['timestamp'], equity_df['bankroll'], 'b-', linewidth=1.5, label='Bankroll')
ax1.axhline(y=10000, color='gray', linestyle='--', alpha=0.5, label='Starting Capital')
ax1.fill_between(equity_df['timestamp'], 10000, equity_df['bankroll'], 
                  where=equity_df['bankroll'] >= 10000, alpha=0.3, color='green')
ax1.fill_between(equity_df['timestamp'], 10000, equity_df['bankroll'], 
                  where=equity_df['bankroll'] < 10000, alpha=0.3, color='red')
ax1.set_ylabel('Bankroll ($)')
ax1.set_title('Equity Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Drawdown
ax2 = axes[1]
running_max = equity_df['bankroll'].cummax()
drawdown = (equity_df['bankroll'] - running_max) / running_max * 100
ax2.fill_between(equity_df['timestamp'], 0, drawdown, color='red', alpha=0.5)
ax2.set_ylabel('Drawdown (%)')
ax2.set_xlabel('Time')
ax2.set_title('Drawdown from Peak')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Max Drawdown: {drawdown.min():.2f}%")
print(f"Current Return: {equity_df['return_pct'].iloc[-1]:.2f}%")

## Overall Performance Summary

In [ ]:
wins = closed[closed['pnl'] > 0]
losses = closed[closed['pnl'] <= 0]

print("=" * 60)
print("OVERALL PERFORMANCE")
print("=" * 60)
print(f"Total Closed Trades: {len(closed)}")
print(f"Wins: {len(wins)} ({len(wins)/len(closed)*100:.1f}%)")
print(f"Losses: {len(losses)} ({len(losses)/len(closed)*100:.1f}%)")
print(f"\nTotal P&L: ${closed['pnl'].sum():.2f}")
print(f"Avg Win: ${wins['pnl'].mean():.2f}")
print(f"Avg Loss: ${losses['pnl'].mean():.2f}")
print(f"Win/Loss Ratio: {abs(wins['pnl'].mean() / losses['pnl'].mean()):.2f}")
print(f"\nLargest Win: ${wins['pnl'].max():.2f}")
print(f"Largest Loss: ${losses['pnl'].min():.2f}")

## Performance by Side (YES vs NO)

In [ ]:
side_stats = closed.groupby('side').agg({
    'pnl': ['count', 'sum', 'mean'],
    'id': lambda x: (closed.loc[x.index, 'pnl'] > 0).sum()
}).round(2)
side_stats.columns = ['Trades', 'Total P&L', 'Avg P&L', 'Wins']
side_stats['Win Rate'] = (side_stats['Wins'] / side_stats['Trades'] * 100).round(1).astype(str) + '%'
side_stats['Total P&L'] = side_stats['Total P&L'].apply(lambda x: f'${x:,.2f}')
side_stats['Avg P&L'] = side_stats['Avg P&L'].apply(lambda x: f'${x:,.2f}')
display(side_stats)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# P&L by side
side_pnl = closed.groupby('side')['pnl'].sum()
colors = ['green' if x > 0 else 'red' for x in side_pnl.values]
axes[0].bar(side_pnl.index, side_pnl.values, color=colors, alpha=0.7)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].set_ylabel('Total P&L ($)')
axes[0].set_title('Total P&L by Side')
for i, (side, pnl) in enumerate(side_pnl.items()):
    axes[0].annotate(f'${pnl:.0f}', (i, pnl), ha='center', 
                     va='bottom' if pnl > 0 else 'top', fontsize=12)

# Win rate by side
win_rates = closed.groupby('side').apply(lambda x: (x['pnl'] > 0).mean() * 100)
axes[1].bar(win_rates.index, win_rates.values, color=['steelblue', 'steelblue'], alpha=0.7)
axes[1].axhline(y=50, color='red', linestyle='--', alpha=0.5, label='50% baseline')
axes[1].set_ylabel('Win Rate (%)')
axes[1].set_title('Win Rate by Side')
axes[1].set_ylim(0, 100)
for i, (side, rate) in enumerate(win_rates.items()):
    axes[1].annotate(f'{rate:.1f}%', (i, rate), ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.show()

## Performance by Edge Size

In [ ]:
# Create edge buckets
closed['edge_bucket'] = pd.cut(closed['edge'], 
                                bins=[0, 0.04, 0.05, 0.07, 0.10, 1.0],
                                labels=['3-4%', '4-5%', '5-7%', '7-10%', '10%+'])

edge_stats = closed.groupby('edge_bucket', observed=True).agg({
    'pnl': ['count', 'sum', 'mean'],
    'entry_price': lambda x: (closed.loc[x.index, 'entry_price'] * closed.loc[x.index, 'size'] / 100).sum(),
    'id': lambda x: (closed.loc[x.index, 'pnl'] > 0).sum()
})
edge_stats.columns = ['Trades', 'Total P&L', 'Avg P&L', 'Capital', 'Wins']
edge_stats['Win Rate'] = (edge_stats['Wins'] / edge_stats['Trades'] * 100).round(1)
edge_stats['ROI'] = (edge_stats['Total P&L'] / edge_stats['Capital'] * 100).round(1)

print(edge_stats[['Trades', 'Win Rate', 'Total P&L', 'ROI']].to_string())

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROI by edge bucket
colors = ['green' if x > 0 else 'red' for x in edge_stats['ROI'].values]
axes[0].bar(range(len(edge_stats)), edge_stats['ROI'].values, color=colors, alpha=0.7)
axes[0].set_xticks(range(len(edge_stats)))
axes[0].set_xticklabels(edge_stats.index)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].set_ylabel('ROI (%)')
axes[0].set_xlabel('Edge Bucket')
axes[0].set_title('ROI by Edge Size')
for i, roi in enumerate(edge_stats['ROI'].values):
    axes[0].annotate(f'{roi:.0f}%', (i, roi), ha='center', 
                     va='bottom' if roi > 0 else 'top', fontsize=10)

# Win rate by edge bucket
axes[1].bar(range(len(edge_stats)), edge_stats['Win Rate'].values, color='steelblue', alpha=0.7)
axes[1].set_xticks(range(len(edge_stats)))
axes[1].set_xticklabels(edge_stats.index)
axes[1].axhline(y=50, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Win Rate (%)')
axes[1].set_xlabel('Edge Bucket')
axes[1].set_title('Win Rate by Edge Size')
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

print("\n⚠️  NOTE: 5-7% edge bucket shows -69% ROI - potential 'death zone'")

## Performance by Entry Price

In [ ]:
# Create price buckets
closed['price_bucket'] = pd.cut(closed['entry_price'], 
                                 bins=[0, 20, 40, 60, 80, 101],
                                 labels=['0-20¢', '20-40¢', '40-60¢', '60-80¢', '80-100¢'])

price_stats = closed.groupby('price_bucket', observed=True).agg({
    'pnl': ['count', 'sum'],
    'id': lambda x: (closed.loc[x.index, 'pnl'] > 0).sum()
})
price_stats.columns = ['Trades', 'Total P&L', 'Wins']
price_stats['Win Rate'] = (price_stats['Wins'] / price_stats['Trades'] * 100).round(1)

# Split by side
for side in ['YES', 'NO']:
    side_data = closed[closed['side'] == side]
    price_stats[f'{side} P&L'] = side_data.groupby('price_bucket', observed=True)['pnl'].sum()

price_stats = price_stats.fillna(0)
print(price_stats.to_string())

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = range(len(price_stats))
width = 0.35

# P&L by price bucket and side
yes_pnl = price_stats['YES P&L'].values
no_pnl = price_stats['NO P&L'].values
axes[0].bar([i - width/2 for i in x], yes_pnl, width, label='YES', 
            color=['green' if v > 0 else 'red' for v in yes_pnl], alpha=0.7)
axes[0].bar([i + width/2 for i in x], no_pnl, width, label='NO',
            color=['green' if v > 0 else 'salmon' for v in no_pnl], alpha=0.7)
axes[0].set_xticks(x)
axes[0].set_xticklabels(price_stats.index)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].set_ylabel('P&L ($)')
axes[0].set_xlabel('Entry Price')
axes[0].set_title('P&L by Entry Price and Side')
axes[0].legend()

# Win rate by price bucket
axes[1].bar(x, price_stats['Win Rate'].values, color='steelblue', alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(price_stats.index)
axes[1].axhline(y=50, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Win Rate (%)')
axes[1].set_xlabel('Entry Price')
axes[1].set_title('Win Rate by Entry Price')
axes[1].set_ylim(0, 110)
for i, rate in enumerate(price_stats['Win Rate'].values):
    axes[1].annotate(f'{rate:.0f}%', (i, rate), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("\n⚠️  NOTE: 40-60¢ bucket has only 22% win rate")
print("✓  NOTE: 80-100¢ bucket has 100% win rate")

## Model Accuracy Analysis

In [ ]:
# Determine if model was correct for each trade
def model_was_correct(row):
    model_says_yes_wins = row['model_prob'] > 0.5
    if row['side'] == 'YES':
        actual_yes_won = row['exit_price'] == 100
    else:
        actual_yes_won = row['exit_price'] == 0
    return model_says_yes_wins == actual_yes_won

closed['model_correct'] = closed.apply(model_was_correct, axis=1)

correct = closed[closed['model_correct']]
wrong = closed[~closed['model_correct']]

print("=" * 60)
print("MODEL ACCURACY")
print("=" * 60)
print(f"Model Correct: {len(correct)}/{len(closed)} ({len(correct)/len(closed)*100:.1f}%)")
print(f"Model Wrong: {len(wrong)}/{len(closed)} ({len(wrong)/len(closed)*100:.1f}%)")
print(f"\nP&L when model correct: ${correct['pnl'].sum():.2f}")
print(f"P&L when model wrong: ${wrong['pnl'].sum():.2f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Model accuracy pie chart
axes[0].pie([len(correct), len(wrong)], labels=['Correct', 'Wrong'], 
            autopct='%1.1f%%', colors=['green', 'red'], alpha=0.7)
axes[0].set_title('Model Prediction Accuracy')

# P&L when correct vs wrong
pnl_data = [correct['pnl'].sum(), wrong['pnl'].sum()]
colors = ['green' if x > 0 else 'red' for x in pnl_data]
axes[1].bar(['Correct', 'Wrong'], pnl_data, color=colors, alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_ylabel('Total P&L ($)')
axes[1].set_title('P&L by Model Accuracy')
for i, pnl in enumerate(pnl_data):
    axes[1].annotate(f'${pnl:.0f}', (i, pnl), ha='center', 
                     va='bottom' if pnl > 0 else 'top', fontsize=12)

plt.tight_layout()
plt.show()

## Trades Where Model Was Wrong (Detail)

In [ ]:
wrong_trades = wrong.sort_values('pnl')[['ticker', 'side', 'entry_price', 'exit_price', 
                                          'size', 'pnl', 'model_prob', 'edge']].copy()
wrong_trades['model_prob'] = (wrong_trades['model_prob'] * 100).round(1).astype(str) + '%'
wrong_trades['edge'] = (wrong_trades['edge'] * 100).round(1).astype(str) + '%'
wrong_trades['pnl'] = wrong_trades['pnl'].apply(lambda x: f'${x:,.2f}')
wrong_trades['entry_price'] = wrong_trades['entry_price'].astype(str) + '¢'
wrong_trades['exit_price'] = wrong_trades['exit_price'].astype(int).astype(str) + '¢'

print("Trades where model predicted wrong outcome (sorted by P&L):")
display(wrong_trades)

## Position Size Analysis

In [ ]:
# Create size buckets
closed['size_bucket'] = pd.cut(closed['size'], 
                                bins=[0, 100, 500, 1000, 10000],
                                labels=['Small (1-99)', 'Medium (100-499)', 
                                        'Large (500-999)', 'XL (1000+)'])

size_stats = closed.groupby('size_bucket', observed=True).agg({
    'pnl': ['count', 'sum', 'mean'],
    'entry_price': lambda x: (closed.loc[x.index, 'entry_price'] * closed.loc[x.index, 'size'] / 100).sum(),
    'id': lambda x: (closed.loc[x.index, 'pnl'] > 0).sum()
})
size_stats.columns = ['Trades', 'Total P&L', 'Avg P&L', 'Capital', 'Wins']
size_stats['Win Rate'] = (size_stats['Wins'] / size_stats['Trades'] * 100).round(1)
size_stats['ROI'] = (size_stats['Total P&L'] / size_stats['Capital'] * 100).round(1)

print(size_stats[['Trades', 'Win Rate', 'Total P&L', 'ROI']].to_string())

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['green' if x > 0 else 'red' for x in size_stats['ROI'].values]
ax.bar(range(len(size_stats)), size_stats['ROI'].values, color=colors, alpha=0.7)
ax.set_xticks(range(len(size_stats)))
ax.set_xticklabels(size_stats.index, rotation=15)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_ylabel('ROI (%)')
ax.set_title('ROI by Position Size')
for i, roi in enumerate(size_stats['ROI'].values):
    ax.annotate(f'{roi:.0f}%', (i, roi), ha='center', 
                va='bottom' if roi > 0 else 'top', fontsize=11)
plt.tight_layout()
plt.show()

## Time of Day Analysis

In [ ]:
# Extract settlement hour
closed['settlement_hour'] = closed['settlement_time'].dt.hour

hour_stats = closed.groupby('settlement_hour').agg({
    'pnl': ['count', 'sum'],
    'id': lambda x: (closed.loc[x.index, 'pnl'] > 0).sum()
})
hour_stats.columns = ['Trades', 'Total P&L', 'Wins']
hour_stats['Win Rate'] = (hour_stats['Wins'] / hour_stats['Trades'] * 100).round(1)

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# P&L by hour
colors = ['green' if x > 0 else 'red' for x in hour_stats['Total P&L'].values]
axes[0].bar(hour_stats.index, hour_stats['Total P&L'].values, color=colors, alpha=0.7)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].set_ylabel('Total P&L ($)')
axes[0].set_title('P&L by Settlement Hour (UTC)')
axes[0].set_xticks(hour_stats.index)

# Win rate by hour
axes[1].bar(hour_stats.index, hour_stats['Win Rate'].values, color='steelblue', alpha=0.7)
axes[1].axhline(y=50, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Win Rate (%)')
axes[1].set_xlabel('Settlement Hour (UTC)')
axes[1].set_title('Win Rate by Settlement Hour (UTC)')
axes[1].set_xticks(hour_stats.index)
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

print("\nHours with negative P&L:")
print(hour_stats[hour_stats['Total P&L'] < 0][['Trades', 'Win Rate', 'Total P&L']])

## Conflicting Positions Analysis
Check if we have YES positions at higher strikes than NO positions (for same settlement), which guarantees at least one loss.

In [ ]:
# Group positions by settlement time
all_positions = positions.copy()
all_positions['settlement_key'] = all_positions['settlement_time'].dt.strftime('%Y%m%d%H')

conflicts = []
for settlement, group in all_positions.groupby('settlement_key'):
    yes_positions = group[group['side'] == 'YES']
    no_positions = group[group['side'] == 'NO']
    
    for _, yes_pos in yes_positions.iterrows():
        for _, no_pos in no_positions.iterrows():
            # Conflict: YES strike > NO strike means there's a gap where both lose
            if yes_pos['strike'] > no_pos['strike']:
                gap = yes_pos['strike'] - no_pos['strike']
                conflicts.append({
                    'settlement': settlement,
                    'yes_strike': yes_pos['strike'],
                    'no_strike': no_pos['strike'],
                    'gap': gap,
                    'yes_cost': yes_pos['entry_price'] * yes_pos['size'] / 100,
                    'no_cost': no_pos['entry_price'] * no_pos['size'] / 100,
                    'yes_pnl': yes_pos['pnl'],
                    'no_pnl': no_pos['pnl']
                })

if conflicts:
    conflict_df = pd.DataFrame(conflicts)
    print(f"Found {len(conflict_df)} conflicting position pairs!")
    print(f"\nThese are cases where YES strike > NO strike, creating a 'death zone'")
    print(f"where if BTC lands between the strikes, BOTH positions lose.\n")
    
    # Show conflicts with completed P&L
    completed_conflicts = conflict_df[(conflict_df['yes_pnl'].notna()) & (conflict_df['no_pnl'].notna())]
    if len(completed_conflicts) > 0:
        completed_conflicts['combined_pnl'] = completed_conflicts['yes_pnl'] + completed_conflicts['no_pnl']
        print("Completed conflicting pairs:")
        display(completed_conflicts[['settlement', 'yes_strike', 'no_strike', 'gap', 
                                      'yes_pnl', 'no_pnl', 'combined_pnl']])
        print(f"\nTotal P&L from conflicting pairs: ${completed_conflicts['combined_pnl'].sum():.2f}")
else:
    print("No conflicting positions found (good!)")

## P&L Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of P&L
axes[0].hist(closed['pnl'], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].axvline(x=closed['pnl'].mean(), color='green', linestyle='--', linewidth=2, label=f'Mean: ${closed["pnl"].mean():.2f}')
axes[0].set_xlabel('P&L ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Trade P&L')
axes[0].legend()

# Cumulative P&L over trades
closed_sorted = closed.sort_values('exit_time')
cumulative_pnl = closed_sorted['pnl'].cumsum()
axes[1].plot(range(len(cumulative_pnl)), cumulative_pnl.values, 'b-', linewidth=1.5)
axes[1].fill_between(range(len(cumulative_pnl)), 0, cumulative_pnl.values,
                      where=cumulative_pnl.values >= 0, alpha=0.3, color='green')
axes[1].fill_between(range(len(cumulative_pnl)), 0, cumulative_pnl.values,
                      where=cumulative_pnl.values < 0, alpha=0.3, color='red')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('Trade Number')
axes[1].set_ylabel('Cumulative P&L ($)')
axes[1].set_title('Cumulative P&L Over Time')

plt.tight_layout()
plt.show()

## Summary & Recommendations

In [ ]:
print("=" * 70)
print("KEY FINDINGS & RECOMMENDATIONS")
print("=" * 70)

print("\n1. SIDE PREFERENCE")
print(f"   - NO positions: {closed[closed['side']=='NO']['pnl'].sum():+.2f} ({(closed[closed['side']=='NO']['pnl']>0).mean()*100:.0f}% win rate)")
print(f"   - YES positions: {closed[closed['side']=='YES']['pnl'].sum():+.2f} ({(closed[closed['side']=='YES']['pnl']>0).mean()*100:.0f}% win rate)")
print("   → Consider favoring NO positions")

print("\n2. EDGE SIZE")
print("   - 5-7% edge bucket has -69% ROI (death zone)")
print("   - 3-4% edge: +12.4% ROI")
print("   - 7-10% edge: +16.3% ROI")
print("   - 10%+ edge: +86.4% ROI")
print("   → Avoid 5-7% edge, prefer <4% or >7%")

print("\n3. ENTRY PRICE")
print("   - 0-20¢: 14% win rate (avoid cheap lottery tickets)")
print("   - 40-60¢: 22% win rate (avoid mid-price YES)")
print("   - 80-100¢: 100% win rate (high probability bets work)")
print("   → Favor high-probability trades (80¢+)")

print("\n4. POSITION SIZE")
print("   - Large (500-999): -8.6% ROI")
print("   - Medium (100-499): +20.7% ROI")
print("   → Consider reducing Kelly fraction further")

print("\n5. MODEL ACCURACY")
print(f"   - Model correct {len(correct)/len(closed)*100:.0f}% of time")
print(f"   - P&L when correct: ${correct['pnl'].sum():+.2f}")
print(f"   - P&L when wrong: ${wrong['pnl'].sum():+.2f}")
print("   → Model is decent but wrong predictions are costly")